api 연동 환경 만들기

In [ ]:
#@title api key 불러오기
import json

# Key는 자신이 발급받은 것으로 변경
json_path = r"D:\내배캠 관련 모음\제미나이 api 키\gemini_config.json"

with open(json_path, "r") as json_file:
    config = json.load(json_file)

In [ ]:
# 1. 패키지 가져오기 (import 방식 변경됨)
from google import genai

# 2. 클라이언트 생성 (기존 configure 대체)
client = genai.Client(api_key=config['GEMINI_API_KEY'])

In [ ]:
# 3. 사용 가능한 모델 목록 확인 (무료 버전에서는 지원이 되지 않는 모델도 있을 수 있음 : limit:0 등 메세지가 뜨는 경우 등)
print("Available Models:")
# client.models.list()를 사용합니다.
for model_info in client.models.list():
    print(model_info.name)

In [ ]:
# gemini-2.5-flash 모델 로드
model_id = "gemini-2.0-flash"

In [ ]:
# 반복 작업 함수화하기
def make_answer(question):
  response = client.models.generate_content(
      model=model_id,
      contents=question
  )
  return response.text

In [ ]:
question3 = "지금 잘 작동중인건가?"

In [ ]:
%%time
make_answer(question3)

In [ ]:
def fmt_seconds(sec: float) -> str:
    try:
        sec = float(sec)
    except Exception:
        return "N/A"
    if not math.isfinite(sec) or sec < 0:
        return "N/A"
    sec = int(sec)
    h = sec // 3600
    m = (sec % 3600) // 60
    s = sec % 60
    if h > 0:
        return f"{h}h {m}m {s}s"
    if m > 0:
        return f"{m}m {s}s"
    return f"{s}s"


1차 분류 작업 진행


최종 버전 3 디벨롭 -> 이중 대기 (GLOBAL_RATE_LIMITER + BASE_THROTTLE_SEC) → 운영 시 BASE_THROTTLE_SEC=0으로 줄이는 방법

In [ ]:
# 이중 대기 (GLOBAL_RATE_LIMITER + BASE_THROTTLE_SEC) → 운영 시 BASE_THROTTLE_SEC=0으로 줄이는 방법



import re
import json
import csv
import time
import math
import hashlib
import threading
import pandas as pd
from pathlib import Path
from typing import Any, Dict, List, Optional

# =========================================================
# 0) 실행 모드
# =========================================================
# test: 1000개 테스트 / prod: 전체 운영
# RUN_MODE = "test"   # <- "prod" 로 바꾸면 전체 운영
RUN_MODE = "prod"
# =========================================================
# 1) 경로/설정
# =========================================================
CSV_PATH = r"D:\내배캠 관련 모음\최종 프로젝트 데이터 셋\아이브 데이터 셋\모든 테이블 concat 버전\광고목록.csv"

OUT_TEST_PATH = r"D:\내배캠 관련 모음\최종 프로젝트 데이터 셋\아이브 데이터 셋\광고 이름 분류\버전 3\광고목록_adsname_domain_labeled_TEST1000.csv"
OUT_PROD_PATH = r"D:\내배캠 관련 모음\최종 프로젝트 데이터 셋\아이브 데이터 셋\광고 이름 분류\버전 3\실전 운용\광고목록_adsname_domain_labeled_ALL.csv"

TEST_N = 1000
BATCH_SIZE = 40

# ---------------------------------------------------------
# ✅ 이중 대기 제거 전략
# - "유일한 대기"는 GLOBAL_RATE_LIMITER.wait() 하나만 둔다
# - 따라서 BASE_THROTTLE_SEC는 운영에서 0으로 둬도 안전(권장)
# ---------------------------------------------------------
if RUN_MODE == "test":
    BASE_THROTTLE_SEC = 0.0   # ✅ 테스트도 전역 limiter만으로 충분 (원하시면 0.5 정도 둘 수도 있음)
else:
    BASE_THROTTLE_SEC = 0.0   # ✅ 운영은 반드시 0 권장(이중대기 제거)

# 재시도/백오프
MAX_RETRIES = 6
MAX_BACKOFF_SEC = 120

# 429 즉시 쿨다운
COOLDOWN_429_SEC = 30

# 자동 분할 최소 단위
MIN_SPLIT_SIZE = 6

# =========================================================
# ✅ (핵심) 전역 Rate Limiter: split 재귀 호출까지 포함하여 RPM 보호
# =========================================================
# 실제로 제한에 걸리는 RPM에 맞게 조정:
# - RPM 15라면 4초
# - 최근 429가 실제로 났으면, 4~5초 권장
GLOBAL_TARGET_RPM = 15
GLOBAL_MIN_INTERVAL_SEC = max(0.1, 60.0 / float(GLOBAL_TARGET_RPM))  # ex) 4.0

class GlobalRateLimiter:
    def __init__(self, min_interval_sec: float):
        self.min_interval_sec = float(min_interval_sec)
        self._lock = threading.Lock()
        self._last_call_ts = 0.0

    def wait(self):
        with self._lock:
            now = time.time()
            if self._last_call_ts <= 0:
                self._last_call_ts = now
                return
            elapsed = now - self._last_call_ts
            if elapsed < self.min_interval_sec:
                time.sleep(self.min_interval_sec - elapsed)
            self._last_call_ts = time.time()

GLOBAL_RATE_LIMITER = GlobalRateLimiter(GLOBAL_MIN_INTERVAL_SEC)

# =========================================================
# (A) 실패 raw 저장 폴더
# =========================================================
OUT_PATH = OUT_TEST_PATH if RUN_MODE == "test" else OUT_PROD_PATH
FAIL_RAW_DIR = Path(OUT_PATH).parent / "_llm_fail_raw"
FAIL_RAW_DIR.mkdir(parents=True, exist_ok=True)

# =========================================================
# (B) 배치 단위 로그 저장 (JSONL + CSV)
# =========================================================
LOG_DIR = Path(OUT_PATH).parent / "_llm_batch_logs"
LOG_DIR.mkdir(parents=True, exist_ok=True)

RUN_ID = time.strftime("%Y%m%d_%H%M%S")
BATCH_LOG_JSONL = LOG_DIR / f"batch_log_{RUN_ID}.jsonl"
BATCH_LOG_CSV = LOG_DIR / f"batch_log_{RUN_ID}.csv"

CSV_FIELDS = [
    "run_id", "batch_id", "event", "attempt",
    "batch_size", "min_size", "split_depth",
    "status", "error_type", "error_message",
    "latency_sec", "cooldown_sec",
    "result_count", "raw_saved_path",
    "names_hash", "names_sample",
    "ts"
]

if not BATCH_LOG_CSV.exists():
    with open(BATCH_LOG_CSV, "w", newline="", encoding="utf-8-sig") as f:
        w = csv.DictWriter(f, fieldnames=CSV_FIELDS)
        w.writeheader()

def _now_ts() -> str:
    return time.strftime("%Y-%m-%d %H:%M:%S")

def _names_hash(names: List[str]) -> str:
    h = hashlib.sha1()
    for n in names:
        h.update((n + "\n").encode("utf-8", errors="ignore"))
    return h.hexdigest()[:16]

def log_batch_row(row: Dict[str, Any]):
    row = dict(row)
    row["ts"] = row.get("ts", _now_ts())

    with open(BATCH_LOG_JSONL, "a", encoding="utf-8") as f:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

    with open(BATCH_LOG_CSV, "a", newline="", encoding="utf-8-sig") as f:
        w = csv.DictWriter(f, fieldnames=CSV_FIELDS)
        w.writerow(row)

# =========================================================
# 2) 라벨 정의(숫자)
# =========================================================
RULE_ENABLED = True

KEYWORDS = {
    1: ["게임","웹툰","드라마","영화","콘텐츠","영상","스트리밍","유튜브","틱톡","sns","소셜","음악","tv","티비"],
    2: ["금융","은행","증권","투자","재테크","대출","캐피탈","카드","보험","적금","예금","포인트","적립","캐시백","리워드","페이","송금","결제","코인","거래소"],
    3: ["헬스","운동","다이어트","건강","통신","요금","알뜰","소개팅","매칭","유틸","예약","교통","지도","서비스","배달","택시","채팅","랜덤채팅","미팅","마사지"],
    4: ["쇼핑","리테일","이커머스","커머스","세일","할인","쿠폰","딜","특가","구매","주문","배송","마켓","스토어","몰","아울렛","멤버십","11번가","쿠팡","네이버쇼핑","ssg","롯데","홈플러스","lf몰","29cm"],
    5: ["test","테스트","tracking","트래킹","qa","dummy","더미","tag","태그","campaign","캠페인","unknown","정체불명","extractor","debug"],
}

def normalize_text(x: str) -> str:
    if x is None:
        return ""
    x = str(x).strip().lower()
    x = re.sub(r"\s+", " ", x)
    return x

def rule_score_counts(name: str) -> Dict[int, int]:
    t = normalize_text(name)
    if not t:
        return {}
    scores: Dict[int, int] = {}
    for label_num, kws in KEYWORDS.items():
        c = 0
        for kw in kws:
            if kw.lower() in t:
                c += 1
        if c > 0:
            scores[label_num] = c
    return scores

def rule_classify_num_tie_to_llm(name: str) -> Optional[int]:
    """
    룰 우선순위: '키워드 매칭 개수 최다' 1개만
    - 최다 1개면 확정
    - 동점/매칭0이면 None -> LLM
    - 빈 문자열은 5
    """
    t = normalize_text(name)
    if not t:
        return 5

    scores = rule_score_counts(name)
    if not scores:
        return None

    max_cnt = max(scores.values())
    winners = [k for k, v in scores.items() if v == max_cnt]
    return winners[0] if len(winners) == 1 else None

# =========================================================
# 3) 유틸
# =========================================================
def fmt_seconds(sec: float) -> str:
    try:
        sec = float(sec)
    except Exception:
        return "N/A"
    if not math.isfinite(sec) or sec < 0:
        return "N/A"
    sec = int(sec)
    h = sec // 3600
    m = (sec % 3600) // 60
    s = sec % 60
    if h > 0:
        return f"{h}h {m}m {s}s"
    if m > 0:
        return f"{m}m {s}s"
    return f"{s}s"

def sanitize_name_for_prompt(x: str, max_len: int = 120) -> str:
    x = str(x).replace("\n", " ").replace("\r", " ").strip()
    if len(x) > max_len:
        x = x[:max_len] + "…"
    return x

def build_batch_prompt_numeric(names: List[str]) -> str:
    items = []
    for i, n in enumerate(names):
        items.append(f'{i}: "{sanitize_name_for_prompt(n)}"')
    joined = "\n".join(items)

    return f"""
당신은 광고명(ads_name) 텍스트만 보고 도메인을 분류합니다.
아래 숫자 라벨 중 하나만 선택하세요.

[숫자 라벨 정의]
1: 엔터테인먼트(게임/콘텐츠/영상/소셜)
2: 금융(금융/보험/재테크/대출/적립/결제/송금)
3: 라이프스타일(생활앱/유틸/헬스/소개팅/통신/예약/교통/서비스)
4: 커머스(쇼핑/리테일/세일/이커머스/마켓/스토어/멤버십/할인/쿠폰)
5: 기타(테스트/트래킹용/정체불명/캠페인 태그성 또는 위에 해당 없음)

[출력 규칙(매우 중요)]
- 반드시 JSON 배열만 출력하세요. (다른 문장/설명/코드블록 금지)
- 배열 길이는 입력 개수와 동일해야 합니다.
- 배열 원소는 반드시 정수 1~5 중 하나여야 합니다.
- i번째 원소는 i번째 입력 광고명에 대응합니다.

[입력 목록]
{joined}
""".strip()

def extract_json_array(text: str) -> List[Any]:
    text = (text or "").strip()
    try:
        obj = json.loads(text)
        if isinstance(obj, list):
            return obj
    except Exception:
        pass

    m = re.search(r"\[.*\]", text, flags=re.DOTALL)
    if not m:
        raise ValueError("No JSON array found in model output.")
    obj = json.loads(m.group(0))
    if not isinstance(obj, list):
        raise ValueError("Parsed JSON is not a list.")
    return obj

def is_rate_limit_429(msg: str) -> bool:
    if not msg:
        return False
    s = msg.lower()
    return ("429" in s) or ("resource_exhausted" in s)

def classify_error_type(err_msg: str) -> str:
    s = (err_msg or "").lower()
    if "batch output size mismatch" in s:
        return "mismatch"
    if is_rate_limit_429(s):
        return "rate_limit_429"
    if "no json array found" in s or "parsed json" in s or "json" in s:
        return "parse_error"
    return "other"

def dump_fail_raw(raw: str, prompt: str, names: List[str], err: Exception, batch_id: int, split_depth: int) -> str:
    ts = time.strftime("%Y%m%d_%H%M%S")
    f = FAIL_RAW_DIR / f"fail_{RUN_ID}_b{batch_id}_d{split_depth}_{ts}_n{len(names)}.json"
    payload = {
        "run_id": RUN_ID,
        "timestamp": ts,
        "batch_id": batch_id,
        "split_depth": split_depth,
        "error": str(err),
        "names_count": len(names),
        "names_sample": names[:10],
        "prompt": prompt,
        "raw": raw,
    }
    f.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
    return str(f)

# =========================================================
# 4) LLM 단일 호출(배치 1회) + 검증
# =========================================================
def gemini_call_once(
    names: List[str],
    make_answer_fn,
    batch_id: int,
    attempt: int,
    split_depth: int
) -> Dict[str, int]:

    prompt = build_batch_prompt_numeric(names)
    names_h = _names_hash(names)
    sample = " | ".join(names[:5])

    t0 = time.time()
    raw = ""
    cooldown = 0

    # ✅ 유일한 대기 지점: split 재귀 포함 "전역" RPM 보호
    GLOBAL_RATE_LIMITER.wait()

    try:
        raw = make_answer_fn(prompt)
        arr = extract_json_array(raw)

        if len(arr) != len(names):
            raise ValueError(f"Batch output size mismatch: got={len(arr)}, expected={len(names)}")

        out: Dict[str, int] = {}
        for i, v in enumerate(arr):
            if isinstance(v, str) and v.strip().isdigit():
                v = int(v.strip())
            if not isinstance(v, int) or v < 1 or v > 5:
                raise ValueError(f"Invalid label value at index {i}: {v}")
            out[names[i]] = v

        latency = time.time() - t0
        log_batch_row({
            "run_id": RUN_ID,
            "batch_id": batch_id,
            "event": "call",
            "attempt": attempt,
            "batch_size": len(names),
            "min_size": MIN_SPLIT_SIZE,
            "split_depth": split_depth,
            "status": "success",
            "error_type": "",
            "error_message": "",
            "latency_sec": round(latency, 3),
            "cooldown_sec": cooldown,
            "result_count": len(out),
            "raw_saved_path": "",
            "names_hash": names_h,
            "names_sample": sample,
        })
        return out

    except Exception as e:
        latency = time.time() - t0
        msg = str(e)
        etype = classify_error_type(msg)

        raw_path = dump_fail_raw(
            raw=raw, prompt=prompt, names=names, err=e,
            batch_id=batch_id, split_depth=split_depth
        )

        # 429면 즉시 30초 쿨다운
        if etype == "rate_limit_429":
            cooldown = COOLDOWN_429_SEC
            log_batch_row({
                "run_id": RUN_ID,
                "batch_id": batch_id,
                "event": "cooldown",
                "attempt": attempt,
                "batch_size": len(names),
                "min_size": MIN_SPLIT_SIZE,
                "split_depth": split_depth,
                "status": "cooldown_applied",
                "error_type": "rate_limit_429",
                "error_message": msg[:240],
                "latency_sec": round(latency, 3),
                "cooldown_sec": cooldown,
                "result_count": 0,
                "raw_saved_path": raw_path,
                "names_hash": names_h,
                "names_sample": sample,
            })
            time.sleep(cooldown)

        log_batch_row({
            "run_id": RUN_ID,
            "batch_id": batch_id,
            "event": "call",
            "attempt": attempt,
            "batch_size": len(names),
            "min_size": MIN_SPLIT_SIZE,
            "split_depth": split_depth,
            "status": "fail",
            "error_type": etype,
            "error_message": msg[:240],
            "latency_sec": round(latency, 3),
            "cooldown_sec": cooldown,
            "result_count": 0,
            "raw_saved_path": raw_path,
            "names_hash": names_h,
            "names_sample": sample,
        })
        raise

# =========================================================
# 5) 자동 분할 + 성공 결과만 합치기
# =========================================================
def classify_with_split(
    names: List[str],
    make_answer_fn,
    batch_id: int,
    split_depth: int = 0
) -> Dict[str, int]:

    names = [n for n in names if isinstance(n, str)]

    for attempt in range(1, MAX_RETRIES + 2):
        try:
            return gemini_call_once(
                names=names,
                make_answer_fn=make_answer_fn,
                batch_id=batch_id,
                attempt=attempt,
                split_depth=split_depth,
            )
        except Exception as e:
            msg = str(e)
            etype = classify_error_type(msg)

            # mismatch면 분할 우선
            if etype == "mismatch":
                if len(names) <= MIN_SPLIT_SIZE:
                    log_batch_row({
                        "run_id": RUN_ID,
                        "batch_id": batch_id,
                        "event": "fallback",
                        "attempt": attempt,
                        "batch_size": len(names),
                        "min_size": MIN_SPLIT_SIZE,
                        "split_depth": split_depth,
                        "status": "fallback_to_5",
                        "error_type": "mismatch",
                        "error_message": msg[:240],
                        "latency_sec": 0,
                        "cooldown_sec": 0,
                        "result_count": len(names),
                        "raw_saved_path": "",
                        "names_hash": _names_hash(names),
                        "names_sample": " | ".join(names[:5]),
                    })
                    return {n: 5 for n in names}

                mid = len(names) // 2
                left = names[:mid]
                right = names[mid:]

                log_batch_row({
                    "run_id": RUN_ID,
                    "batch_id": batch_id,
                    "event": "split",
                    "attempt": attempt,
                    "batch_size": len(names),
                    "min_size": MIN_SPLIT_SIZE,
                    "split_depth": split_depth,
                    "status": "split_into_two",
                    "error_type": "mismatch",
                    "error_message": msg[:240],
                    "latency_sec": 0,
                    "cooldown_sec": 0,
                    "result_count": 0,
                    "raw_saved_path": "",
                    "names_hash": _names_hash(names),
                    "names_sample": " | ".join(names[:5]),
                })

                out: Dict[str, int] = {}
                out.update(classify_with_split(left, make_answer_fn, batch_id, split_depth + 1))
                out.update(classify_with_split(right, make_answer_fn, batch_id, split_depth + 1))
                return out

            # 429는 call_once에서 이미 쿨다운 수행.
            backoff = min(MAX_BACKOFF_SEC, (attempt * 0.5))
            log_batch_row({
                "run_id": RUN_ID,
                "batch_id": batch_id,
                "event": "retry_wait",
                "attempt": attempt,
                "batch_size": len(names),
                "min_size": MIN_SPLIT_SIZE,
                "split_depth": split_depth,
                "status": "waiting",
                "error_type": etype,
                "error_message": msg[:240],
                "latency_sec": 0,
                "cooldown_sec": backoff,
                "result_count": 0,
                "raw_saved_path": "",
                "names_hash": _names_hash(names),
                "names_sample": " | ".join(names[:5]),
            })
            time.sleep(backoff)

    log_batch_row({
        "run_id": RUN_ID,
        "batch_id": batch_id,
        "event": "fallback",
        "attempt": MAX_RETRIES + 1,
        "batch_size": len(names),
        "min_size": MIN_SPLIT_SIZE,
        "split_depth": split_depth,
        "status": "fallback_to_5",
        "error_type": "final_fail",
        "error_message": "",
        "latency_sec": 0,
        "cooldown_sec": 0,
        "result_count": len(names),
        "raw_saved_path": "",
        "names_hash": _names_hash(names),
        "names_sample": " | ".join(names[:5]),
    })
    return {n: 5 for n in names}

# =========================================================
# 6) 실행(테스트/운영 공용)
# =========================================================
def run_domain_labeling(make_answer_fn):
    in_path = Path(CSV_PATH)
    if not in_path.exists():
        raise FileNotFoundError(f"입력 CSV가 존재하지 않습니다: {CSV_PATH}")

    df = pd.read_csv(CSV_PATH, usecols=["ads_name"], dtype={"ads_name": "string"}, low_memory=False)

    if RUN_MODE == "test":
        df_work = df.head(TEST_N).copy()  # 원본 순서 유지
    else:
        df_work = df.copy()

    df_work["ads_name"] = df_work["ads_name"].fillna("").astype(str)

    names_all = df_work["ads_name"].tolist()
    unique_names = pd.Series(names_all).drop_duplicates().tolist()  # 중복 제거(순서 유지)

    name_to_label: Dict[str, int] = {}
    llm_targets: List[str] = []

    if RULE_ENABLED:
        for n in unique_names:
            if not n.strip():
                name_to_label[n] = 5
                continue
            r = rule_classify_num_tie_to_llm(n)
            if r is None:
                llm_targets.append(n)
            else:
                name_to_label[n] = r
    else:
        llm_targets = [n for n in unique_names if n.strip()]

    print(f"[MODE] {RUN_MODE}")
    print(f"[WORK] rows={len(df_work)} | unique={len(unique_names)} | rule_done={len(name_to_label)} | llm_targets={len(llm_targets)}")
    print(f"[CONF] BATCH_SIZE={BATCH_SIZE} | BASE_THROTTLE_SEC={BASE_THROTTLE_SEC}")
    print(f"[CONF] GLOBAL_TARGET_RPM={GLOBAL_TARGET_RPM} -> GLOBAL_MIN_INTERVAL_SEC={GLOBAL_MIN_INTERVAL_SEC:.2f}s (split 포함 전역 보호)")
    print(f"[LOG]  jsonl={BATCH_LOG_JSONL}")
    print(f"[LOG]  csv ={BATCH_LOG_CSV}")
    print(f"[FAIL] raw_dir={FAIL_RAW_DIR}")

    processed = 0
    t0 = time.time()
    batch_id = 0

    for b_start in range(0, len(llm_targets), BATCH_SIZE):
        batch = llm_targets[b_start:b_start + BATCH_SIZE]

        batch_out = classify_with_split(batch, make_answer_fn, batch_id=batch_id, split_depth=0)
        name_to_label.update(batch_out)

        processed += len(batch)
        batch_id += 1

        elapsed = time.time() - t0
        speed = processed / elapsed if elapsed > 0 else 0.0
        remain = len(llm_targets) - processed
        eta_sec = remain / speed if speed > 0 else float("inf")

        if processed <= (BATCH_SIZE * 3) or processed % (BATCH_SIZE * 10) == 0:
            print(f"[LLM] {processed}/{len(llm_targets)} | speed={speed:.2f} items/s | ETA={fmt_seconds(eta_sec)}")

        # ✅ (선택) 추가 대기: 전역 limiter가 이미 있으므로 기본은 0 (이중 대기 방지)
        if BASE_THROTTLE_SEC and BASE_THROTTLE_SEC > 0:
            time.sleep(BASE_THROTTLE_SEC)

    df_work["domain_label"] = df_work["ads_name"].map(lambda x: int(name_to_label.get(x, 5)))

    out_path = Path(OUT_PATH)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    df_work[["ads_name", "domain_label"]].to_csv(out_path, index=False, encoding="utf-8-sig")

    print(f"\n✅ 저장 완료: {out_path} | rows={len(df_work)}")
    print("[라벨 분포]")
    print(df_work["domain_label"].value_counts().sort_index())

    return df_work

# =========================================================
# 실행
# =========================================================
# df_labeled = run_domain_labeling(make_answer)


In [ ]:
# 실전 운영 실행 코드

# RUN_MODE = "prod"
df_all = run_domain_labeling(make_answer)


2차 분류 작업 진행

버전 4 동욱님 분류 결과와 비교하여 불일치 하는 광고 이름만 재분류

In [ ]:
# =========================================================
# ✅ 최종 코드: 두 CSV 라벨 불일치 행만 재분류 → 2번 CSV 갱신 저장
# - 1번 CSV: ads_name, domain_label_1
# - 2번 CSV: ads_name, domain_label
# - ads_name 순서/텍스트가 두 파일에서 완전히 동일하다는 전제(검증 포함)
# - 불일치 ads_name만 LLM 재분류(배치+자동분할+전역 RPM+로그+fail raw 저장)
# - 최종 저장: 2번 CSV의 행 순서를 유지하며 domain_label만 업데이트
# =========================================================

import re
import json
import csv
import time
import math
import hashlib
import threading
import pandas as pd
from pathlib import Path
from typing import Any, Dict, List, Optional

# =========================================================
# 0) 경로/모드 설정
# =========================================================
RUN_MODE = "prod"   # "test" or "prod"



# ✅ 입력 CSV
CSV1_PATH = r"D:\내배캠 관련 모음\최종 프로젝트 데이터 셋\아이브 데이터 셋\광고목록_adsname_domain_labeled_ALL_adsname_fixed_20260220_011654.csv"   # 1번(팀원) CSV
CSV2_PATH = r"D:\내배캠 관련 모음\최종 프로젝트 데이터 셋\아이브 데이터 셋\광고 이름 분류\버전 3\실전 운용\광고목록_adsname_domain_labeled_ALL_adsname_fixed_20260220_011351.csv"     # 2번(본인) CSV

# ✅ 출력(테스트/운영)
OUT_TEST_PATH = r"D:\내배캠 관련 모음\최종 프로젝트 데이터 셋\아이브 데이터 셋\광고 이름 분류\버전 4\테스트 버전 2\my_result_reconciled_TEST1000.csv"
OUT_PROD_PATH = r"D:\내배캠 관련 모음\최종 프로젝트 데이터 셋\아이브 데이터 셋\광고 이름 분류\버전 6\실전 운용\my_result_reconciled_ALL.csv"

# ✅ 테스트 개수
TEST_N = 1000

# ✅ 배치 크기
BATCH_SIZE = 40

# =========================================================
# 1) 운영 파라미터
# =========================================================
# 이중 대기 제거: 전역 limiter만 사용
BASE_THROTTLE_SEC = 0.0

# 재시도/백오프
MAX_RETRIES = 6
MAX_BACKOFF_SEC = 120

# 429 즉시 쿨다운
COOLDOWN_429_SEC = 30

# 자동 분할 최소 단위
MIN_SPLIT_SIZE = 6

# =========================================================
# ✅ 전역 Rate Limiter (split 재귀 호출까지 포함)
# =========================================================
# 실제로 제한에 걸렸던 RPM에 맞춰 설정하세요.
# (예: RPM 15 => 4.0초)
GLOBAL_TARGET_RPM = 15
GLOBAL_MIN_INTERVAL_SEC = max(0.1, 60.0 / float(GLOBAL_TARGET_RPM))  # ex) 4.0

class GlobalRateLimiter:
    def __init__(self, min_interval_sec: float):
        self.min_interval_sec = float(min_interval_sec)
        self._lock = threading.Lock()
        self._last_call_ts = 0.0

    def wait(self):
        with self._lock:
            now = time.time()
            if self._last_call_ts <= 0:
                self._last_call_ts = now
                return
            elapsed = now - self._last_call_ts
            if elapsed < self.min_interval_sec:
                time.sleep(self.min_interval_sec - elapsed)
            self._last_call_ts = time.time()

GLOBAL_RATE_LIMITER = GlobalRateLimiter(GLOBAL_MIN_INTERVAL_SEC)

# =========================================================
# 2) 출력/로그/실패 raw 저장 폴더 준비
# =========================================================
OUT_PATH = OUT_TEST_PATH if RUN_MODE == "test" else OUT_PROD_PATH
OUT_PATH = str(OUT_PATH)

FAIL_RAW_DIR = Path(OUT_PATH).parent / "_llm_fail_raw"
FAIL_RAW_DIR.mkdir(parents=True, exist_ok=True)

LOG_DIR = Path(OUT_PATH).parent / "_llm_batch_logs"
LOG_DIR.mkdir(parents=True, exist_ok=True)

RUN_ID = time.strftime("%Y%m%d_%H%M%S")
BATCH_LOG_JSONL = LOG_DIR / f"batch_log_{RUN_ID}.jsonl"
BATCH_LOG_CSV = LOG_DIR / f"batch_log_{RUN_ID}.csv"

CSV_FIELDS = [
    "run_id", "batch_id", "event", "attempt",
    "batch_size", "min_size", "split_depth",
    "status", "error_type", "error_message",
    "latency_sec", "cooldown_sec",
    "result_count", "raw_saved_path",
    "names_hash", "names_sample",
    "ts"
]

if not BATCH_LOG_CSV.exists():
    with open(BATCH_LOG_CSV, "w", newline="", encoding="utf-8-sig") as f:
        w = csv.DictWriter(f, fieldnames=CSV_FIELDS)
        w.writeheader()

def _now_ts() -> str:
    return time.strftime("%Y-%m-%d %H:%M:%S")

def _names_hash(names: List[str]) -> str:
    h = hashlib.sha1()
    for n in names:
        h.update((str(n) + "\n").encode("utf-8", errors="ignore"))
    return h.hexdigest()[:16]

def log_batch_row(row: Dict[str, Any]):
    row = dict(row)
    row["ts"] = row.get("ts", _now_ts())

    with open(BATCH_LOG_JSONL, "a", encoding="utf-8") as f:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

    with open(BATCH_LOG_CSV, "a", newline="", encoding="utf-8-sig") as f:
        w = csv.DictWriter(f, fieldnames=CSV_FIELDS)
        w.writerow(row)

# =========================================================
# 3) 룰 기반(선택) - 정확도 보정에 도움될 수 있으나
#    ✅ "불일치 재분류" 목적이면 RULE_ENABLED=False 권장
# =========================================================
RULE_ENABLED = False  # ✅ 불일치만 LLM로 재분류하려면 False 권장

KEYWORDS = {
    1: ["게임","웹툰","드라마","영화","콘텐츠","영상","스트리밍","유튜브","틱톡","sns","소셜","음악","tv","티비"],
    2: ["금융","은행","증권","투자","재테크","대출","캐피탈","카드","보험","적금","예금","포인트","적립","캐시백","리워드","페이","송금","결제","코인","거래소"],
    3: ["헬스","운동","다이어트","건강","통신","요금","알뜰","소개팅","매칭","유틸","예약","교통","지도","서비스","배달","택시","채팅","랜덤채팅","미팅","마사지"],
    4: ["쇼핑","리테일","이커머스","커머스","세일","할인","쿠폰","딜","특가","구매","주문","배송","마켓","스토어","몰","아울렛","멤버십","11번가","쿠팡","네이버쇼핑","ssg","롯데","홈플러스","lf몰","29cm"],
    5: ["test","테스트","tracking","트래킹","qa","dummy","더미","tag","태그","campaign","캠페인","unknown","정체불명","extractor","debug"],
}

def normalize_text(x: str) -> str:
    if x is None:
        return ""
    x = str(x).strip().lower()
    x = re.sub(r"\s+", " ", x)
    return x

def rule_score_counts(name: str) -> Dict[int, int]:
    t = normalize_text(name)
    if not t:
        return {}
    scores: Dict[int, int] = {}
    for label_num, kws in KEYWORDS.items():
        c = 0
        for kw in kws:
            if kw.lower() in t:
                c += 1
        if c > 0:
            scores[label_num] = c
    return scores

def rule_classify_num_tie_to_llm(name: str) -> Optional[int]:
    """
    룰 우선순위: '키워드 매칭 개수 최다' 1개만
    - 최다 1개면 확정
    - 동점/매칭0이면 None -> LLM
    - 빈 문자열은 5
    """
    t = normalize_text(name)
    if not t:
        return 5

    scores = rule_score_counts(name)
    if not scores:
        return None

    max_cnt = max(scores.values())
    winners = [k for k, v in scores.items() if v == max_cnt]
    return winners[0] if len(winners) == 1 else None

# =========================================================
# 4) 프롬프트/파서/에러 분류
# =========================================================
def sanitize_name_for_prompt(x: str, max_len: int = 120) -> str:
    x = str(x).replace("\n", " ").replace("\r", " ").strip()
    if len(x) > max_len:
        x = x[:max_len] + "…"
    return x

def build_batch_prompt_numeric(names: List[str]) -> str:
    items = []
    for i, n in enumerate(names):
        items.append(f'{i}: "{sanitize_name_for_prompt(n)}"')
    joined = "\n".join(items)

    return f"""
당신은 광고명(ads_name) 텍스트만 보고 도메인을 분류합니다.
아래 숫자 라벨 중 하나만 선택하세요.

[숫자 라벨 정의]
1: 엔터테인먼트
- 대상: 콘텐츠 소비/오락 중심 서비스
- 대표 신호: 게임/웹툰/만화/드라마/영화/OTT/스트리밍/방송/아이돌/팬덤/음악/공연/티켓/유튜브/틱톡/아프리카/트위치/커뮤니티/SNS
- 예시 표현: "쿠키", "무료충전(콘텐츠)", "시청/구독", "팬클럽", "스밍", "플레이", "캐릭터", "스킨", "보상(게임)"
- 주의: “결제/적립”이 있어도 목적이 콘텐츠/게임이면 1이 우선

2: 금융
- 대상: 돈(자산/대출/보험/투자/결제) 자체가 핵심인 서비스
- 대표 신호: 은행/증권/투자/재테크/대출/캐피탈/카드/보험/적금/예금/연금/환전/송금/간편결제/페이/결제수단/현금/캐시백/포인트(금융상품)
- 예시 표현: "한도", "금리", "이자", "신용", "심사", "보험료", "보장", "청구", "카드발급", "계좌", "송금", "결제"
- 주의: 쇼핑 결제(주문/배송) 맥락이면 4가 우선

3: 라이프스타일
- 대상: 생활 편의/일상 서비스(커머스·금융·엔터를 제외한 나머지 앱 서비스)
- 대표 신호: 헬스/운동/다이어트/의료/병원/약/통신/요금/알뜰폰/예약/교통/지도/택시/대리/배달(서비스 이용)/숙박/여행/학습/자기계발/채팅/소개팅/매칭/유틸(청소, 최적화, 보안 등)
- 예시 표현: "예약", "호출", "근처", "경로", "요금제", "스케줄", "상담", "매칭", "운동기록"
- 주의: “배달”이 음식/생활 서비스면 3, 쇼핑몰 상품 배송이면 4

4: 커머스
- 대상: 상품/거래(구매·주문·배송·특가·쿠폰·장바구니)가 핵심인 서비스
- 대표 신호: 쇼핑/마켓/스토어/몰/아울렛/특가/딜/쿠폰/할인/세일/장바구니/주문/구매/결제(주문결제)/배송/택배/구독(상품정기)/브랜드/리뷰/포인트(쇼핑적립)
- 예시 표현: "오늘만특가", "무료배송", "장바구니", "구매후기", "정기배송", "상품", "브랜드관", "쿠폰팩"
- 주의: ‘포인트/적립’이 있어도 맥락이 쇼핑/주문이면 4가 우선

5: 기타
- 대상: 테스트/트래킹/태그/캠페인용/정체불명 또는 위 1~4에 근거가 부족한 경우
- 대표 신호: test/qa/dummy/debug/tracking/tag/campaign/utm/광고ID/코드성 문자열/의미 없는 조합
- 규칙: 애매하면 5 (억지로 1~4를 고르지 말 것)

[판정 우선순위 규칙]
- (1) ‘구매/주문/배송/장바구니/쿠폰/특가/몰/마켓’ 등 거래 신호가 있으면 → 4(커머스) 우선
- (2) ‘대출/보험/금리/한도/계좌/송금/카드발급’ 등 금융상품 신호가 있으면 → 2(금융) 우선
- (3) ‘게임/웹툰/드라마/스트리밍/팬덤/음악/티켓’ 등 오락 신호가 있으면 → 1(엔터) 우선
- (4) 위에 해당하지 않고 생활 서비스 신호(예약/교통/통신/헬스/매칭 등)가 있으면 → 3(라이프스타일)
- (5) 근거 부족/코드성/테스트성/애매하면 → 5(기타)

[출력 규칙(매우 중요)]
- 반드시 JSON 배열만 출력하세요. (다른 문장/설명/코드블록 금지)
- 배열 길이는 입력 개수와 동일해야 합니다.
- 배열 원소는 반드시 정수 1~5 중 하나여야 합니다.
- i번째 원소는 i번째 입력 광고명에 대응합니다.


[입력 목록]
{joined}
""".strip()

def extract_json_array(text: str) -> List[Any]:
    text = (text or "").strip()
    try:
        obj = json.loads(text)
        if isinstance(obj, list):
            return obj
    except Exception:
        pass

    m = re.search(r"\[.*\]", text, flags=re.DOTALL)
    if not m:
        raise ValueError("No JSON array found in model output.")
    obj = json.loads(m.group(0))
    if not isinstance(obj, list):
        raise ValueError("Parsed JSON is not a list.")
    return obj

def is_rate_limit_429(msg: str) -> bool:
    if not msg:
        return False
    s = msg.lower()
    return ("429" in s) or ("resource_exhausted" in s)

def classify_error_type(err_msg: str) -> str:
    s = (err_msg or "").lower()
    if "batch output size mismatch" in s:
        return "mismatch"
    if is_rate_limit_429(s):
        return "rate_limit_429"
    if "no json array found" in s or "parsed json" in s or "json" in s:
        return "parse_error"
    return "other"

def dump_fail_raw(raw: str, prompt: str, names: List[str], err: Exception, batch_id: int, split_depth: int) -> str:
    ts = time.strftime("%Y%m%d_%H%M%S")
    f = FAIL_RAW_DIR / f"fail_{RUN_ID}_b{batch_id}_d{split_depth}_{ts}_n{len(names)}.json"
    payload = {
        "run_id": RUN_ID,
        "timestamp": ts,
        "batch_id": batch_id,
        "split_depth": split_depth,
        "error": str(err),
        "names_count": len(names),
        "names_sample": names[:10],
        "prompt": prompt,
        "raw": raw,
    }
    f.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
    return str(f)

# =========================================================
# 5) LLM 단일 호출 + 자동분할
# =========================================================
def gemini_call_once(
    names: List[str],
    make_answer_fn,
    batch_id: int,
    attempt: int,
    split_depth: int
) -> Dict[str, int]:

    prompt = build_batch_prompt_numeric(names)
    names_h = _names_hash(names)
    sample = " | ".join(names[:5])

    t0 = time.time()
    raw = ""
    cooldown = 0

    # ✅ 유일한 대기 지점: split 재귀 포함 전역 RPM 보호
    GLOBAL_RATE_LIMITER.wait()

    try:
        raw = make_answer_fn(prompt)
        arr = extract_json_array(raw)

        if len(arr) != len(names):
            raise ValueError(f"Batch output size mismatch: got={len(arr)}, expected={len(names)}")

        out: Dict[str, int] = {}
        for i, v in enumerate(arr):
            if isinstance(v, str) and v.strip().isdigit():
                v = int(v.strip())
            if not isinstance(v, int) or v < 1 or v > 5:
                raise ValueError(f"Invalid label value at index {i}: {v}")
            out[names[i]] = v

        latency = time.time() - t0
        log_batch_row({
            "run_id": RUN_ID,
            "batch_id": batch_id,
            "event": "call",
            "attempt": attempt,
            "batch_size": len(names),
            "min_size": MIN_SPLIT_SIZE,
            "split_depth": split_depth,
            "status": "success",
            "error_type": "",
            "error_message": "",
            "latency_sec": round(latency, 3),
            "cooldown_sec": cooldown,
            "result_count": len(out),
            "raw_saved_path": "",
            "names_hash": names_h,
            "names_sample": sample,
        })
        return out

    except Exception as e:
        latency = time.time() - t0
        msg = str(e)
        etype = classify_error_type(msg)

        raw_path = dump_fail_raw(
            raw=raw, prompt=prompt, names=names, err=e,
            batch_id=batch_id, split_depth=split_depth
        )

        # 429면 즉시 30초 쿨다운
        if etype == "rate_limit_429":
            cooldown = COOLDOWN_429_SEC
            log_batch_row({
                "run_id": RUN_ID,
                "batch_id": batch_id,
                "event": "cooldown",
                "attempt": attempt,
                "batch_size": len(names),
                "min_size": MIN_SPLIT_SIZE,
                "split_depth": split_depth,
                "status": "cooldown_applied",
                "error_type": "rate_limit_429",
                "error_message": msg[:240],
                "latency_sec": round(latency, 3),
                "cooldown_sec": cooldown,
                "result_count": 0,
                "raw_saved_path": raw_path,
                "names_hash": names_h,
                "names_sample": sample,
            })
            time.sleep(cooldown)

        log_batch_row({
            "run_id": RUN_ID,
            "batch_id": batch_id,
            "event": "call",
            "attempt": attempt,
            "batch_size": len(names),
            "min_size": MIN_SPLIT_SIZE,
            "split_depth": split_depth,
            "status": "fail",
            "error_type": etype,
            "error_message": msg[:240],
            "latency_sec": round(latency, 3),
            "cooldown_sec": cooldown,
            "result_count": 0,
            "raw_saved_path": raw_path,
            "names_hash": names_h,
            "names_sample": sample,
        })
        raise

def classify_with_split(
    names: List[str],
    make_answer_fn,
    batch_id: int,
    split_depth: int = 0
) -> Dict[str, int]:

    names = [n for n in names if isinstance(n, str)]

    for attempt in range(1, MAX_RETRIES + 2):
        try:
            return gemini_call_once(
                names=names,
                make_answer_fn=make_answer_fn,
                batch_id=batch_id,
                attempt=attempt,
                split_depth=split_depth,
            )
        except Exception as e:
            msg = str(e)
            etype = classify_error_type(msg)

            # mismatch면 분할 우선
            if etype == "mismatch":
                if len(names) <= MIN_SPLIT_SIZE:
                    log_batch_row({
                        "run_id": RUN_ID,
                        "batch_id": batch_id,
                        "event": "fallback",
                        "attempt": attempt,
                        "batch_size": len(names),
                        "min_size": MIN_SPLIT_SIZE,
                        "split_depth": split_depth,
                        "status": "fallback_to_5",
                        "error_type": "mismatch",
                        "error_message": msg[:240],
                        "latency_sec": 0,
                        "cooldown_sec": 0,
                        "result_count": len(names),
                        "raw_saved_path": "",
                        "names_hash": _names_hash(names),
                        "names_sample": " | ".join(names[:5]),
                    })
                    return {n: 5 for n in names}

                mid = len(names) // 2
                left = names[:mid]
                right = names[mid:]

                log_batch_row({
                    "run_id": RUN_ID,
                    "batch_id": batch_id,
                    "event": "split",
                    "attempt": attempt,
                    "batch_size": len(names),
                    "min_size": MIN_SPLIT_SIZE,
                    "split_depth": split_depth,
                    "status": "split_into_two",
                    "error_type": "mismatch",
                    "error_message": msg[:240],
                    "latency_sec": 0,
                    "cooldown_sec": 0,
                    "result_count": 0,
                    "raw_saved_path": "",
                    "names_hash": _names_hash(names),
                    "names_sample": " | ".join(names[:5]),
                })

                out: Dict[str, int] = {}
                out.update(classify_with_split(left, make_answer_fn, batch_id, split_depth + 1))
                out.update(classify_with_split(right, make_answer_fn, batch_id, split_depth + 1))
                return out

            # 429는 call_once에서 이미 쿨다운 수행.
            backoff = min(MAX_BACKOFF_SEC, (attempt * 0.5))
            log_batch_row({
                "run_id": RUN_ID,
                "batch_id": batch_id,
                "event": "retry_wait",
                "attempt": attempt,
                "batch_size": len(names),
                "min_size": MIN_SPLIT_SIZE,
                "split_depth": split_depth,
                "status": "waiting",
                "error_type": etype,
                "error_message": msg[:240],
                "latency_sec": 0,
                "cooldown_sec": backoff,
                "result_count": 0,
                "raw_saved_path": "",
                "names_hash": _names_hash(names),
                "names_sample": " | ".join(names[:5]),
            })
            time.sleep(backoff)

    log_batch_row({
        "run_id": RUN_ID,
        "batch_id": batch_id,
        "event": "fallback",
        "attempt": MAX_RETRIES + 1,
        "batch_size": len(names),
        "min_size": MIN_SPLIT_SIZE,
        "split_depth": split_depth,
        "status": "fallback_to_5",
        "error_type": "final_fail",
        "error_message": "",
        "latency_sec": 0,
        "cooldown_sec": 0,
        "result_count": len(names),
        "raw_saved_path": "",
        "names_hash": _names_hash(names),
        "names_sample": " | ".join(names[:5]),
    })
    return {n: 5 for n in names}

# =========================================================
# 6) 핵심 실행: 불일치만 재분류 후 2번 CSV 갱신 저장
# =========================================================
def reconcile_and_relabel(make_answer_fn):
    # 파일 존재 체크
    p1 = Path(CSV1_PATH)
    p2 = Path(CSV2_PATH)
    if not p1.exists():
        raise FileNotFoundError(f"1번 CSV가 존재하지 않습니다: {CSV1_PATH}")
    if not p2.exists():
        raise FileNotFoundError(f"2번 CSV가 존재하지 않습니다: {CSV2_PATH}")

    # 로드
    df1 = pd.read_csv(CSV1_PATH, dtype={"ads_name": "string"}, low_memory=False)
    df2 = pd.read_csv(CSV2_PATH, dtype={"ads_name": "string"}, low_memory=False)

    # test면 첫 TEST_N행만
    if RUN_MODE == "test":
        df1 = df1.head(TEST_N).copy()
        df2 = df2.head(TEST_N).copy()

    # 필수 컬럼 체크
    need1 = {"ads_name", "domain_label_1"}
    need2 = {"ads_name", "domain_label"}
    if not need1.issubset(df1.columns):
        raise ValueError(f"1번 CSV에 필요한 컬럼이 없습니다. 필요={need1}, 현재={set(df1.columns)}")
    if not need2.issubset(df2.columns):
        raise ValueError(f"2번 CSV에 필요한 컬럼이 없습니다. 필요={need2}, 현재={set(df2.columns)}")

    # ads_name 완전 동일(순서+텍스트) 검증
    a1 = df1["ads_name"].fillna("").astype(str).tolist()
    a2 = df2["ads_name"].fillna("").astype(str).tolist()
    if a1 != a2:
        raise ValueError("두 CSV의 ads_name이 (순서/텍스트까지) 완전히 일치하지 않습니다. 전제 조건이 깨졌습니다.")

    # 라벨 비교(순서 기반)
    l1 = pd.to_numeric(df1["domain_label_1"], errors="coerce").fillna(-1).astype(int)
    l2 = pd.to_numeric(df2["domain_label"],   errors="coerce").fillna(-1).astype(int)

    diff_mask = (l1 != l2)
    diff_idx = df2.index[diff_mask].tolist()

    print(f"[MODE] {RUN_MODE}")
    print(f"[SIZE] rows={len(df2)} | mismatch_rows={len(diff_idx)}")
    print(f"[CONF] BATCH_SIZE={BATCH_SIZE} | BASE_THROTTLE_SEC={BASE_THROTTLE_SEC}")
    print(f"[CONF] GLOBAL_TARGET_RPM={GLOBAL_TARGET_RPM} -> interval={GLOBAL_MIN_INTERVAL_SEC:.2f}s (split 포함 전역 보호)")
    print(f"[LOG]  jsonl={BATCH_LOG_JSONL}")
    print(f"[LOG]  csv ={BATCH_LOG_CSV}")
    print(f"[FAIL] raw_dir={FAIL_RAW_DIR}")
    print(f"[OUT]  out_path={OUT_PATH}")

    # 불일치가 없으면 그대로 저장
    if len(diff_idx) == 0:
        out_path = Path(OUT_PATH)
        out_path.parent.mkdir(parents=True, exist_ok=True)
        df2.to_csv(out_path, index=False, encoding="utf-8-sig")
        print(f"✅ 불일치 없음. 그대로 저장 완료: {out_path}")
        return df2

    # 불일치 ads_name만 추출(중복 제거)
    diff_names_all = df2.loc[diff_idx, "ads_name"].fillna("").astype(str).tolist()
    diff_unique_names = pd.Series(diff_names_all).drop_duplicates().tolist()

    name_to_label: Dict[str, int] = {}
    llm_targets: List[str] = []

    if RULE_ENABLED:
        for nm in diff_unique_names:
            if not nm.strip():
                name_to_label[nm] = 5
                continue
            r = rule_classify_num_tie_to_llm(nm)
            if r is None:
                llm_targets.append(nm)
            else:
                name_to_label[nm] = r
    else:
        llm_targets = [nm for nm in diff_unique_names if nm.strip()]

    print(f"[TARGET] diff_unique={len(diff_unique_names)} | rule_done={len(name_to_label)} | llm_targets={len(llm_targets)}")

    # LLM 배치 처리(불일치만)
    processed = 0
    t0 = time.time()
    batch_id = 0

    for b_start in range(0, len(llm_targets), BATCH_SIZE):
        batch = llm_targets[b_start:b_start + BATCH_SIZE]
        batch_out = classify_with_split(batch, make_answer_fn, batch_id=batch_id, split_depth=0)
        name_to_label.update(batch_out)

        processed += len(batch)
        batch_id += 1

        elapsed = time.time() - t0
        speed = processed / elapsed if elapsed > 0 else 0.0
        remain = len(llm_targets) - processed
        eta_sec = remain / speed if speed > 0 else float("inf")

        if processed <= (BATCH_SIZE * 3) or processed % (BATCH_SIZE * 10) == 0:
            print(f"[LLM] {processed}/{len(llm_targets)} | speed={speed:.2f} items/s | ETA≈{int(eta_sec//3600)}h {int((eta_sec%3600)//60)}m {int(eta_sec%60)}s")

        if BASE_THROTTLE_SEC and BASE_THROTTLE_SEC > 0:
            time.sleep(BASE_THROTTLE_SEC)

    # diff 검수용 파일 저장
    diff_report_path = Path(OUT_PATH).with_name(f"diff_reclassified_{RUN_ID}.csv")
    df_diff = df2.loc[diff_idx, ["ads_name"]].copy()
    df_diff["domain_label_1"] = l1.loc[diff_idx].values
    df_diff["domain_label_old"] = l2.loc[diff_idx].values
    df_diff["domain_label_new"] = df_diff["ads_name"].map(lambda x: int(name_to_label.get(str(x), 5)))
    diff_report_path.parent.mkdir(parents=True, exist_ok=True)
    df_diff.to_csv(diff_report_path, index=False, encoding="utf-8-sig")
    print(f"✅ diff 검수용 저장: {diff_report_path} | rows={len(df_diff)}")

    # 2번 CSV 갱신(불일치 행만)
    df2.loc[diff_idx, "domain_label"] = df2.loc[diff_idx, "ads_name"].map(lambda x: int(name_to_label.get(str(x), 5))).astype(int)

    # 최종 저장
    out_path = Path(OUT_PATH)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    df2.to_csv(out_path, index=False, encoding="utf-8-sig")
    print(f"\n✅ 최종 저장 완료: {out_path} | rows={len(df2)}")
    print("[라벨 분포]")
    print(pd.to_numeric(df2["domain_label"], errors="coerce").fillna(-1).astype(int).value_counts().sort_index())

    return df2

# =========================================================
# 실행
# =========================================================
# ✅ 이미 정의해둔 make_answer(prompt)->text 함수를 그대로 사용합니다.
# df_fixed = reconcile_and_relabel(make_answer)


In [ ]:
# 실전용 RUN_MODE = "prod"로 바꾸고 실행

df_fixed = reconcile_and_relabel(make_answer)


3차 분류 작업

기타 5번 값만 재분류

In [ ]:
# =========================================================
# ✅ 최종 코드(라벨=5 기타만 재분류): 1개 CSV에서 domain_label==5만 LLM 재분류 → domain_label 갱신 저장
# - 입력 CSV: ads_name, domain_label (최종 결과 파일 1개)
# - 대상: domain_label == 5 인 행들만
# - 처리: 5들 중 유니크 ads_name만 LLM 재분류(배치+자동분할+전역 RPM+로그+fail raw 저장)
# - 결과: 예측이 5면 유지, 1~4면 해당 ads_name 행들의 domain_label을 업데이트
# - 저장: 전체 CSV를 새 파일로 저장 + 검수용 리포트 저장
# =========================================================

import re
import json
import csv
import time
import hashlib
import threading
import pandas as pd
from pathlib import Path
from typing import Any, Dict, List, Optional

# =========================================================
# 0) 경로/모드 설정
# =========================================================
RUN_MODE = "prod"   # "test" or "prod"

# ✅ 입력: 최종 출력된 "단일" CSV (기존 결과 파일)
INPUT_CSV_PATH = r"D:\내배캠 관련 모음\최종 프로젝트 데이터 셋\아이브 데이터 셋\광고 이름 분류\버전 6\실전 운용\my_result_reconciled_ALL.csv"

# ✅ 출력(테스트/운영)
OUT_TEST_PATH = r"D:\내배캠 관련 모음\최종 프로젝트 데이터 셋\아이브 데이터 셋\광고 이름 분류\버전 7\테스트\my_result_reclassified_label5_TEST1000.csv"
OUT_PROD_PATH = r"D:\내배캠 관련 모음\최종 프로젝트 데이터 셋\아이브 데이터 셋\광고 이름 분류\버전 6\3차 기타 5번 재분류\my_result_reclassified_label5_ALL.csv"

# ✅ 테스트 개수(상단 N행만 대상으로 빠르게 확인)
TEST_N = 1000

# ✅ 배치 크기
BATCH_SIZE = 40

# ✅ 라벨 컬럼명 (기본: domain_label)
LABEL_COL = "domain_label"
NAME_COL = "ads_name"

# =========================================================
# 1) 운영 파라미터
# =========================================================
BASE_THROTTLE_SEC = 0.0

MAX_RETRIES = 6
MAX_BACKOFF_SEC = 120
COOLDOWN_429_SEC = 30
MIN_SPLIT_SIZE = 6

# =========================================================
# ✅ 전역 Rate Limiter (split 재귀 호출까지 포함)
# =========================================================
GLOBAL_TARGET_RPM = 15
GLOBAL_MIN_INTERVAL_SEC = max(0.1, 60.0 / float(GLOBAL_TARGET_RPM))

class GlobalRateLimiter:
    def __init__(self, min_interval_sec: float):
        self.min_interval_sec = float(min_interval_sec)
        self._lock = threading.Lock()
        self._last_call_ts = 0.0

    def wait(self):
        with self._lock:
            now = time.time()
            if self._last_call_ts <= 0:
                self._last_call_ts = now
                return
            elapsed = now - self._last_call_ts
            if elapsed < self.min_interval_sec:
                time.sleep(self.min_interval_sec - elapsed)
            self._last_call_ts = time.time()

GLOBAL_RATE_LIMITER = GlobalRateLimiter(GLOBAL_MIN_INTERVAL_SEC)

# =========================================================
# 2) 출력/로그/실패 raw 저장 폴더 준비
# =========================================================
OUT_PATH = OUT_TEST_PATH if RUN_MODE == "test" else OUT_PROD_PATH
OUT_PATH = str(OUT_PATH)

FAIL_RAW_DIR = Path(OUT_PATH).parent / "_llm_fail_raw"
FAIL_RAW_DIR.mkdir(parents=True, exist_ok=True)

LOG_DIR = Path(OUT_PATH).parent / "_llm_batch_logs"
LOG_DIR.mkdir(parents=True, exist_ok=True)

RUN_ID = time.strftime("%Y%m%d_%H%M%S")
BATCH_LOG_JSONL = LOG_DIR / f"batch_log_label5_{RUN_ID}.jsonl"
BATCH_LOG_CSV = LOG_DIR / f"batch_log_label5_{RUN_ID}.csv"

CSV_FIELDS = [
    "run_id", "batch_id", "event", "attempt",
    "batch_size", "min_size", "split_depth",
    "status", "error_type", "error_message",
    "latency_sec", "cooldown_sec",
    "result_count", "raw_saved_path",
    "names_hash", "names_sample",
    "ts"
]

if not BATCH_LOG_CSV.exists():
    with open(BATCH_LOG_CSV, "w", newline="", encoding="utf-8-sig") as f:
        w = csv.DictWriter(f, fieldnames=CSV_FIELDS)
        w.writeheader()

def _now_ts() -> str:
    return time.strftime("%Y-%m-%d %H:%M:%S")

def _names_hash(names: List[str]) -> str:
    h = hashlib.sha1()
    for n in names:
        h.update((str(n) + "\n").encode("utf-8", errors="ignore"))
    return h.hexdigest()[:16]

def log_batch_row(row: Dict[str, Any]):
    row = dict(row)
    row["ts"] = row.get("ts", _now_ts())

    with open(BATCH_LOG_JSONL, "a", encoding="utf-8") as f:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

    with open(BATCH_LOG_CSV, "a", newline="", encoding="utf-8-sig") as f:
        w = csv.DictWriter(f, fieldnames=CSV_FIELDS)
        w.writerow(row)

# =========================================================
# 3) 룰 기반(선택) - 원하시면 True로 켜서 LLM 호출량을 줄일 수 있음
# =========================================================
RULE_ENABLED = False

KEYWORDS = {
    1: ["게임","웹툰","드라마","영화","콘텐츠","영상","스트리밍","유튜브","틱톡","sns","소셜","음악","tv","티비"],
    2: ["금융","은행","증권","투자","재테크","대출","캐피탈","카드","보험","적금","예금","포인트","적립","캐시백","리워드","페이","송금","결제","코인","거래소"],
    3: ["헬스","운동","다이어트","건강","통신","요금","알뜰","소개팅","매칭","유틸","예약","교통","지도","서비스","배달","택시","채팅","랜덤채팅","미팅","마사지"],
    4: ["쇼핑","리테일","이커머스","커머스","세일","할인","쿠폰","딜","특가","구매","주문","배송","마켓","스토어","몰","아울렛","멤버십","11번가","쿠팡","네이버쇼핑","ssg","롯데","홈플러스","lf몰","29cm"],
    5: ["test","테스트","tracking","트래킹","qa","dummy","더미","tag","태그","campaign","캠페인","unknown","정체불명","extractor","debug"],
}

def normalize_text(x: str) -> str:
    if x is None:
        return ""
    x = str(x).strip().lower()
    x = re.sub(r"\s+", " ", x)
    return x

def rule_score_counts(name: str) -> Dict[int, int]:
    t = normalize_text(name)
    if not t:
        return {}
    scores: Dict[int, int] = {}
    for label_num, kws in KEYWORDS.items():
        c = 0
        for kw in kws:
            if kw.lower() in t:
                c += 1
        if c > 0:
            scores[label_num] = c
    return scores

def rule_classify_num_tie_to_llm(name: str) -> Optional[int]:
    t = normalize_text(name)
    if not t:
        return 5
    scores = rule_score_counts(name)
    if not scores:
        return None
    max_cnt = max(scores.values())
    winners = [k for k, v in scores.items() if v == max_cnt]
    return winners[0] if len(winners) == 1 else None

# =========================================================
# 4) 프롬프트/파서/에러 분류
# =========================================================
def sanitize_name_for_prompt(x: str, max_len: int = 120) -> str:
    x = str(x).replace("\n", " ").replace("\r", " ").strip()
    if len(x) > max_len:
        x = x[:max_len] + "…"
    return x

def build_batch_prompt_numeric(names: List[str]) -> str:
    items = []
    for i, n in enumerate(names):
        items.append(f'{i}: "{sanitize_name_for_prompt(n)}"')
    joined = "\n".join(items)

    return f"""
당신은 광고명(ads_name) 텍스트만 보고 도메인을 분류합니다.
아래 숫자 라벨 중 하나만 선택하세요.

[숫자 라벨 정의]
1: 엔터테인먼트
- 대상: 콘텐츠 소비/오락 중심 서비스
- 대표 신호: 게임/웹툰/만화/드라마/영화/OTT/스트리밍/방송/아이돌/팬덤/음악/공연/티켓/유튜브/틱톡/아프리카/트위치/커뮤니티/SNS
- 예시 표현: "쿠키", "무료충전(콘텐츠)", "시청/구독", "팬클럽", "스밍", "플레이", "캐릭터", "스킨", "보상(게임)"
- 주의: “결제/적립”이 있어도 목적이 콘텐츠/게임이면 1이 우선

2: 금융
- 대상: 돈(자산/대출/보험/투자/결제) 자체가 핵심인 서비스
- 대표 신호: 은행/증권/투자/재테크/대출/캐피탈/카드/보험/적금/예금/연금/환전/송금/간편결제/페이/결제수단/현금/캐시백/포인트(금융상품)
- 예시 표현: "한도", "금리", "이자", "신용", "심사", "보험료", "보장", "청구", "카드발급", "계좌", "송금", "결제"
- 주의: 쇼핑 결제(주문/배송) 맥락이면 4가 우선

3: 라이프스타일
- 대상: 생활 편의/일상 서비스(커머스·금융·엔터를 제외한 나머지 앱 서비스)
- 대표 신호: 헬스/운동/다이어트/의료/병원/약/통신/요금/알뜰폰/예약/교통/지도/택시/대리/배달(서비스 이용)/숙박/여행/학습/자기계발/채팅/소개팅/매칭/유틸(청소, 최적화, 보안 등)
- 예시 표현: "예약", "호출", "근처", "경로", "요금제", "스케줄", "상담", "매칭", "운동기록"
- 주의: “배달”이 음식/생활 서비스면 3, 쇼핑몰 상품 배송이면 4

4: 커머스
- 대상: 상품/거래(구매·주문·배송·특가·쿠폰·장바구니)가 핵심인 서비스
- 대표 신호: 쇼핑/마켓/스토어/몰/아울렛/특가/딜/쿠폰/할인/세일/장바구니/주문/구매/결제(주문결제)/배송/택배/구독(상품정기)/브랜드/리뷰/포인트(쇼핑적립)
- 예시 표현: "오늘만특가", "무료배송", "장바구니", "구매후기", "정기배송", "상품", "브랜드관", "쿠폰팩"
- 주의: ‘포인트/적립’이 있어도 맥락이 쇼핑/주문이면 4가 우선

5: 기타
- 대상: 테스트/트래킹/태그/캠페인용/정체불명 또는 위 1~4에 근거가 부족한 경우
- 대표 신호: test/qa/dummy/debug/tracking/tag/campaign/utm/광고ID/코드성 문자열/의미 없는 조합
- 규칙: 애매하면 5 (억지로 1~4를 고르지 말 것)

[판정 우선순위 규칙]
- (1) ‘구매/주문/배송/장바구니/쿠폰/특가/몰/마켓’ 등 거래 신호가 있으면 → 4(커머스) 우선
- (2) ‘대출/보험/금리/한도/계좌/송금/카드발급’ 등 금융상품 신호가 있으면 → 2(금융) 우선
- (3) ‘게임/웹툰/드라마/스트리밍/팬덤/음악/티켓’ 등 오락 신호가 있으면 → 1(엔터) 우선
- (4) 위에 해당하지 않고 생활 서비스 신호(예약/교통/통신/헬스/매칭 등)가 있으면 → 3(라이프스타일)
- (5) 근거 부족/코드성/테스트성/애매하면 → 5(기타)

[출력 규칙(매우 중요)]
- 반드시 JSON 배열만 출력하세요. (다른 문장/설명/코드블록 금지)
- 배열 길이는 입력 개수와 동일해야 합니다.
- 배열 원소는 반드시 정수 1~5 중 하나여야 합니다.
- i번째 원소는 i번째 입력 광고명에 대응합니다.

[입력 목록]
{joined}
""".strip()

def extract_json_array(text: str) -> List[Any]:
    text = (text or "").strip()
    try:
        obj = json.loads(text)
        if isinstance(obj, list):
            return obj
    except Exception:
        pass

    m = re.search(r"\[.*\]", text, flags=re.DOTALL)
    if not m:
        raise ValueError("No JSON array found in model output.")
    obj = json.loads(m.group(0))
    if not isinstance(obj, list):
        raise ValueError("Parsed JSON is not a list.")
    return obj

def is_rate_limit_429(msg: str) -> bool:
    if not msg:
        return False
    s = msg.lower()
    return ("429" in s) or ("resource_exhausted" in s)

def classify_error_type(err_msg: str) -> str:
    s = (err_msg or "").lower()
    if "batch output size mismatch" in s:
        return "mismatch"
    if is_rate_limit_429(s):
        return "rate_limit_429"
    if "no json array found" in s or "parsed json" in s or "json" in s:
        return "parse_error"
    return "other"

def dump_fail_raw(raw: str, prompt: str, names: List[str], err: Exception, batch_id: int, split_depth: int) -> str:
    ts = time.strftime("%Y%m%d_%H%M%S")
    f = FAIL_RAW_DIR / f"fail_{RUN_ID}_b{batch_id}_d{split_depth}_{ts}_n{len(names)}.json"
    payload = {
        "run_id": RUN_ID,
        "timestamp": ts,
        "batch_id": batch_id,
        "split_depth": split_depth,
        "error": str(err),
        "names_count": len(names),
        "names_sample": names[:10],
        "prompt": prompt,
        "raw": raw,
    }
    f.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
    return str(f)

# =========================================================
# 5) LLM 단일 호출 + 자동분할
# =========================================================
def gemini_call_once(
    names: List[str],
    make_answer_fn,
    batch_id: int,
    attempt: int,
    split_depth: int
) -> Dict[str, int]:

    prompt = build_batch_prompt_numeric(names)
    names_h = _names_hash(names)
    sample = " | ".join(names[:5])

    t0 = time.time()
    raw = ""
    cooldown = 0

    GLOBAL_RATE_LIMITER.wait()

    try:
        raw = make_answer_fn(prompt)
        arr = extract_json_array(raw)

        if len(arr) != len(names):
            raise ValueError(f"Batch output size mismatch: got={len(arr)}, expected={len(names)}")

        out: Dict[str, int] = {}
        for i, v in enumerate(arr):
            if isinstance(v, str) and v.strip().isdigit():
                v = int(v.strip())
            if not isinstance(v, int) or v < 1 or v > 5:
                raise ValueError(f"Invalid label value at index {i}: {v}")
            out[names[i]] = v

        latency = time.time() - t0
        log_batch_row({
            "run_id": RUN_ID,
            "batch_id": batch_id,
            "event": "call",
            "attempt": attempt,
            "batch_size": len(names),
            "min_size": MIN_SPLIT_SIZE,
            "split_depth": split_depth,
            "status": "success",
            "error_type": "",
            "error_message": "",
            "latency_sec": round(latency, 3),
            "cooldown_sec": cooldown,
            "result_count": len(out),
            "raw_saved_path": "",
            "names_hash": names_h,
            "names_sample": sample,
        })
        return out

    except Exception as e:
        latency = time.time() - t0
        msg = str(e)
        etype = classify_error_type(msg)

        raw_path = dump_fail_raw(
            raw=raw, prompt=prompt, names=names, err=e,
            batch_id=batch_id, split_depth=split_depth
        )

        if etype == "rate_limit_429":
            cooldown = COOLDOWN_429_SEC
            log_batch_row({
                "run_id": RUN_ID,
                "batch_id": batch_id,
                "event": "cooldown",
                "attempt": attempt,
                "batch_size": len(names),
                "min_size": MIN_SPLIT_SIZE,
                "split_depth": split_depth,
                "status": "cooldown_applied",
                "error_type": "rate_limit_429",
                "error_message": msg[:240],
                "latency_sec": round(latency, 3),
                "cooldown_sec": cooldown,
                "result_count": 0,
                "raw_saved_path": raw_path,
                "names_hash": names_h,
                "names_sample": sample,
            })
            time.sleep(cooldown)

        log_batch_row({
            "run_id": RUN_ID,
            "batch_id": batch_id,
            "event": "call",
            "attempt": attempt,
            "batch_size": len(names),
            "min_size": MIN_SPLIT_SIZE,
            "split_depth": split_depth,
            "status": "fail",
            "error_type": etype,
            "error_message": msg[:240],
            "latency_sec": round(latency, 3),
            "cooldown_sec": cooldown,
            "result_count": 0,
            "raw_saved_path": raw_path,
            "names_hash": names_h,
            "names_sample": sample,
        })
        raise

def classify_with_split(
    names: List[str],
    make_answer_fn,
    batch_id: int,
    split_depth: int = 0
) -> Dict[str, int]:

    names = [n for n in names if isinstance(n, str)]

    for attempt in range(1, MAX_RETRIES + 2):
        try:
            return gemini_call_once(
                names=names,
                make_answer_fn=make_answer_fn,
                batch_id=batch_id,
                attempt=attempt,
                split_depth=split_depth,
            )
        except Exception as e:
            msg = str(e)
            etype = classify_error_type(msg)

            if etype == "mismatch":
                if len(names) <= MIN_SPLIT_SIZE:
                    log_batch_row({
                        "run_id": RUN_ID,
                        "batch_id": batch_id,
                        "event": "fallback",
                        "attempt": attempt,
                        "batch_size": len(names),
                        "min_size": MIN_SPLIT_SIZE,
                        "split_depth": split_depth,
                        "status": "fallback_to_5",
                        "error_type": "mismatch",
                        "error_message": msg[:240],
                        "latency_sec": 0,
                        "cooldown_sec": 0,
                        "result_count": len(names),
                        "raw_saved_path": "",
                        "names_hash": _names_hash(names),
                        "names_sample": " | ".join(names[:5]),
                    })
                    return {n: 5 for n in names}

                mid = len(names) // 2
                left = names[:mid]
                right = names[mid:]

                log_batch_row({
                    "run_id": RUN_ID,
                    "batch_id": batch_id,
                    "event": "split",
                    "attempt": attempt,
                    "batch_size": len(names),
                    "min_size": MIN_SPLIT_SIZE,
                    "split_depth": split_depth,
                    "status": "split_into_two",
                    "error_type": "mismatch",
                    "error_message": msg[:240],
                    "latency_sec": 0,
                    "cooldown_sec": 0,
                    "result_count": 0,
                    "raw_saved_path": "",
                    "names_hash": _names_hash(names),
                    "names_sample": " | ".join(names[:5]),
                })

                out: Dict[str, int] = {}
                out.update(classify_with_split(left, make_answer_fn, batch_id, split_depth + 1))
                out.update(classify_with_split(right, make_answer_fn, batch_id, split_depth + 1))
                return out

            backoff = min(MAX_BACKOFF_SEC, (attempt * 0.5))
            log_batch_row({
                "run_id": RUN_ID,
                "batch_id": batch_id,
                "event": "retry_wait",
                "attempt": attempt,
                "batch_size": len(names),
                "min_size": MIN_SPLIT_SIZE,
                "split_depth": split_depth,
                "status": "waiting",
                "error_type": etype,
                "error_message": msg[:240],
                "latency_sec": 0,
                "cooldown_sec": backoff,
                "result_count": 0,
                "raw_saved_path": "",
                "names_hash": _names_hash(names),
                "names_sample": " | ".join(names[:5]),
            })
            time.sleep(backoff)

    log_batch_row({
        "run_id": RUN_ID,
        "batch_id": batch_id,
        "event": "fallback",
        "attempt": MAX_RETRIES + 1,
        "batch_size": len(names),
        "min_size": MIN_SPLIT_SIZE,
        "split_depth": split_depth,
        "status": "fallback_to_5",
        "error_type": "final_fail",
        "error_message": "",
        "latency_sec": 0,
        "cooldown_sec": 0,
        "result_count": len(names),
        "raw_saved_path": "",
        "names_hash": _names_hash(names),
        "names_sample": " | ".join(names[:5]),
    })
    return {n: 5 for n in names}

# =========================================================
# 6) 핵심 실행: label==5만 재분류 후 CSV 갱신 저장
# =========================================================
def reclassify_only_label5(make_answer_fn):
    p = Path(INPUT_CSV_PATH)
    if not p.exists():
        raise FileNotFoundError(f"입력 CSV가 존재하지 않습니다: {INPUT_CSV_PATH}")

    df = pd.read_csv(INPUT_CSV_PATH, dtype={NAME_COL: "string"}, low_memory=False)

    if RUN_MODE == "test":
        df = df.head(TEST_N).copy()

    # 필수 컬럼 체크
    need = {NAME_COL, LABEL_COL}
    if not need.issubset(df.columns):
        raise ValueError(f"입력 CSV에 필요한 컬럼이 없습니다. 필요={need}, 현재={set(df.columns)}")

    # 라벨을 int로 안전 변환
    labels_old = pd.to_numeric(df[LABEL_COL], errors="coerce").fillna(-1).astype(int)

    # label==5 대상
    idx5 = df.index[labels_old == 5].tolist()

    print(f"[MODE] {RUN_MODE}")
    print(f"[SIZE] rows={len(df):,} | label5_rows={len(idx5):,}")
    print(f"[CONF] BATCH_SIZE={BATCH_SIZE} | GLOBAL_TARGET_RPM={GLOBAL_TARGET_RPM} -> interval={GLOBAL_MIN_INTERVAL_SEC:.2f}s")
    print(f"[LOG]  jsonl={BATCH_LOG_JSONL}")
    print(f"[LOG]  csv ={BATCH_LOG_CSV}")
    print(f"[FAIL] raw_dir={FAIL_RAW_DIR}")
    print(f"[OUT]  out_path={OUT_PATH}")

    if len(idx5) == 0:
        out_path = Path(OUT_PATH)
        out_path.parent.mkdir(parents=True, exist_ok=True)
        df.to_csv(out_path, index=False, encoding="utf-8-sig")
        print("✅ label==5 행이 없어 재분류 없이 그대로 저장했습니다.")
        return df

    # label==5 ads_name만 (유니크)
    names5_all = df.loc[idx5, NAME_COL].fillna("").astype(str).tolist()
    names5_unique = pd.Series(names5_all).drop_duplicates().tolist()

    # 빈 문자열 제외(빈 문자열은 그대로 5 유지)
    name_to_label: Dict[str, int] = {"": 5}
    llm_targets: List[str] = []

    if RULE_ENABLED:
        for nm in names5_unique:
            if not str(nm).strip():
                name_to_label[str(nm)] = 5
                continue
            r = rule_classify_num_tie_to_llm(str(nm))
            if r is None:
                llm_targets.append(str(nm))
            else:
                name_to_label[str(nm)] = int(r)
    else:
        llm_targets = [str(nm) for nm in names5_unique if str(nm).strip()]

    print(f"[TARGET] label5_unique={len(names5_unique):,} | rule_done={len(name_to_label):,} | llm_targets={len(llm_targets):,}")

    # LLM 배치 처리
    processed = 0
    t0 = time.time()
    batch_id = 0

    for b_start in range(0, len(llm_targets), BATCH_SIZE):
        batch = llm_targets[b_start:b_start + BATCH_SIZE]
        batch_out = classify_with_split(batch, make_answer_fn, batch_id=batch_id, split_depth=0)
        name_to_label.update(batch_out)

        processed += len(batch)
        batch_id += 1

        elapsed = time.time() - t0
        speed = processed / elapsed if elapsed > 0 else 0.0
        remain = len(llm_targets) - processed
        eta_sec = remain / speed if speed > 0 else float("inf")

        if processed <= (BATCH_SIZE * 3) or processed % (BATCH_SIZE * 10) == 0:
            print(f"[LLM] {processed}/{len(llm_targets)} | speed={speed:.2f} items/s | ETA≈{int(eta_sec//3600)}h {int((eta_sec%3600)//60)}m {int(eta_sec%60)}s")

        if BASE_THROTTLE_SEC and BASE_THROTTLE_SEC > 0:
            time.sleep(BASE_THROTTLE_SEC)

    # 업데이트: label==5였던 행들만 새 라벨로 교체(여전히 5면 유지)
    new_labels_for_rows = df.loc[idx5, NAME_COL].fillna("").astype(str).map(lambda x: int(name_to_label.get(x, 5)))
    df_new = df.copy()
    df_new.loc[idx5, LABEL_COL] = new_labels_for_rows.astype(int)

    # 검수 리포트 저장(변경된 것만/전체)
    report_all_path = Path(OUT_PATH).with_name(f"label5_reclassified_ALL_{RUN_ID}.csv")
    report_changed_path = Path(OUT_PATH).with_name(f"label5_reclassified_CHANGED_{RUN_ID}.csv")

    df_report = pd.DataFrame({
        NAME_COL: df.loc[idx5, NAME_COL].fillna("").astype(str).values,
        "domain_label_old": labels_old.loc[idx5].values,
        "domain_label_new": new_labels_for_rows.astype(int).values
    })
    df_report["changed"] = (df_report["domain_label_old"] != df_report["domain_label_new"])

    report_all_path.parent.mkdir(parents=True, exist_ok=True)
    df_report.to_csv(report_all_path, index=False, encoding="utf-8-sig")
    df_report.loc[df_report["changed"]].to_csv(report_changed_path, index=False, encoding="utf-8-sig")

    # 최종 저장
    out_path = Path(OUT_PATH)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    df_new.to_csv(out_path, index=False, encoding="utf-8-sig")

    # 요약 출력
    changed_rows = int(df_report["changed"].sum())
    stayed5_rows = int((df_report["domain_label_new"] == 5).sum())
    moved_rows = len(df_report) - stayed5_rows

    print("\n✅ label==5 재분류 완료")
    print(f"- label5_rows(처리 대상 행): {len(df_report):,}")
    print(f"- changed_rows(라벨 변경 행): {changed_rows:,}")
    print(f"- stayed_5_rows(여전히 5): {stayed5_rows:,}")
    print(f"- moved_to_1_4_rows(1~4로 이동): {moved_rows:,}")
    print(f"- report_all     : {report_all_path}")
    print(f"- report_changed : {report_changed_path}")
    print(f"- saved_output   : {out_path}")

    print("\n[라벨 분포 - 변경 후]")
    print(pd.to_numeric(df_new[LABEL_COL], errors="coerce").fillna(-1).astype(int).value_counts().sort_index())

    return df_new

# =========================================================
# 실행
# =========================================================
# ✅ 이미 정의해둔 make_answer(prompt)->text 함수를 그대로 사용합니다.
# df_updated = reclassify_only_label5(make_answer)

In [ ]:
# 실전용 RUN_MODE = "prod"로 바꾸고 실행

df_updated = reclassify_only_label5(make_answer)


최종 라벨링 파일과 원본 파일 결합

In [ ]:
import pandas as pd
from pathlib import Path

# ==============================
# 0) 경로 설정 (환경에 맞게 수정)
# ==============================
CSV1_PATH = r"D:\내배캠 관련 모음\최종 프로젝트 데이터 셋\아이브 데이터 셋\모든 테이블 concat 버전\광고목록.csv"                 # 1번: 원본(순서 기준)
CSV2_PATH = r"D:\내배캠 관련 모음\최종 프로젝트 데이터 셋\아이브 데이터 셋\광고 이름 분류\버전 6\3차 기타 5번 재분류\my_result_reclassified_label5_ALL.csv"              # 2번: domain_label 보유
OUT_PATH  = r"D:\내배캠 관련 모음\최종 프로젝트 데이터 셋\아이브 데이터 셋\광고 이름 분류\버전 6\3차 기타 5번 재분류\원본에_라벨붙인_결과.csv"      # 출력

LABEL_COL = "domain_label"  # 2번 파일에서 가져올 컬럼명

# ==============================
# 1) 로드
# ==============================
df1 = pd.read_csv(CSV1_PATH, low_memory=False)
df2 = pd.read_csv(CSV2_PATH, low_memory=False)

# ==============================
# 2) 기본 검증 (순서 동일 전제이지만, 최소 안전장치)
# ==============================
if LABEL_COL not in df2.columns:
    raise ValueError(f"2번 CSV에 '{LABEL_COL}' 컬럼이 없습니다. 현재 컬럼={list(df2.columns)}")

if len(df1) != len(df2):
    raise ValueError(f"행 수가 다릅니다: df1={len(df1):,}, df2={len(df2):,} (순서 기반 결합 불가)")

# (선택) domain_label 숫자화(문자/결측 대비)
labels = pd.to_numeric(df2[LABEL_COL], errors="coerce").fillna(-1).astype(int)

# ==============================
# 3) 1번 원본에 라벨 컬럼 붙이기 (행 순서 그대로)
# ==============================
df_out = df1.copy()
df_out[LABEL_COL] = labels.values  # 순서대로 붙임

# ==============================
# 4) 저장
# ==============================
out_path = Path(OUT_PATH)
out_path.parent.mkdir(parents=True, exist_ok=True)
df_out.to_csv(out_path, index=False, encoding="utf-8-sig")

print(f"✅ 완료: 1번 원본에 '{LABEL_COL}' 컬럼을 붙여 저장했습니다.")
print(f" - rows: {len(df_out):,}")
print(f" - saved: {out_path}")
print("[라벨 분포]")
print(df_out[LABEL_COL].value_counts(dropna=False).sort_index())

제대로 잘 붙였는지 확인 코드

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import time

# ==============================
# 경로 (환경에 맞게 수정)
# ==============================
FINAL_MERGED_PATH = r"D:\내배캠 관련 모음\최종 프로젝트 데이터 셋\아이브 데이터 셋\광고 이름 분류\버전 6\3차 기타 5번 재분류\원본에_라벨붙인_결과.csv"  # 최종 결합된 파일(df_out 저장본)
LABEL_SOURCE_PATH = r"D:\내배캠 관련 모음\최종 프로젝트 데이터 셋\아이브 데이터 셋\광고 이름 분류\버전 6\3차 기타 5번 재분류\my_result_reclassified_label5_ALL.csv"            # 결합에 사용했던 2번 라벨링 파일(df2)

LABEL_COL = "domain_label"
# (선택) 검증 리포트 저장 경로
REPORT_DIR = Path(r"D:\내배캠 관련 모음\최종 프로젝트 데이터 셋\아이브 데이터 셋\광고 이름 분류\버전 6\3차 기타 5번 재분류\검증리포트")
REPORT_DIR.mkdir(parents=True, exist_ok=True)

# ==============================
# 1) 로드
# ==============================
df_out = pd.read_csv(FINAL_MERGED_PATH, low_memory=False)
df2    = pd.read_csv(LABEL_SOURCE_PATH, low_memory=False)

# ==============================
# 2) 기본 체크
# ==============================
if LABEL_COL not in df_out.columns:
    raise ValueError(f"최종 파일에 '{LABEL_COL}' 컬럼이 없습니다.")
if LABEL_COL not in df2.columns:
    raise ValueError(f"라벨링 파일에 '{LABEL_COL}' 컬럼이 없습니다.")

if len(df_out) != len(df2):
    raise ValueError(f"행 수가 다릅니다: final={len(df_out):,}, label_source={len(df2):,}")

# ==============================
# 3) 순서+값 완전 동일성 검증
#    - 숫자/문자 혼합 대비: to_numeric로 표준화(-1은 결측/비정상 fallback)
# ==============================
s_out = pd.to_numeric(df_out[LABEL_COL], errors="coerce").fillna(-1).astype(int).to_numpy()
s_src = pd.to_numeric(df2[LABEL_COL],    errors="coerce").fillna(-1).astype(int).to_numpy()

match_mask = (s_out == s_src)
all_match = bool(match_mask.all())

print("=================================================")
print("[검증] domain_label 순서/값 동일성")
print(f" - rows: {len(s_out):,}")
print(f" - all_match: {all_match}")

# ==============================
# 4) 불일치가 있으면 상세 출력 + 리포트 저장
# ==============================
if not all_match:
    mismatch_idx = np.where(~match_mask)[0]
    print(f" - mismatch_count: {len(mismatch_idx):,}")
    first = int(mismatch_idx[0])
    print(f" - first_mismatch_idx: {first}")

    # 주변 5행(앞뒤 2행) 샘플
    lo = max(0, first - 2)
    hi = min(len(s_out), first + 3)
    print("   [around first mismatch: i-2 ~ i+2]")
    for i in range(lo, hi):
        flag = " <-- mismatch" if s_out[i] != s_src[i] else ""
        print(f"   idx={i} | final={s_out[i]} | src={s_src[i]}{flag}")

    # 리포트 저장
    ts = time.strftime("%Y%m%d_%H%M%S")
    out_csv = REPORT_DIR / f"domain_label_order_mismatch_{ts}.csv"

    # (선택) ads_name 같이 저장하면 추적이 쉬움 (있는 경우에만)
    cols = {}
    cols["idx"] = mismatch_idx
    cols["final_domain_label"] = s_out[mismatch_idx]
    cols["src_domain_label"]   = s_src[mismatch_idx]

    if "ads_name" in df_out.columns:
        cols["final_ads_name"] = df_out.loc[mismatch_idx, "ads_name"].astype(str).values
    if "ads_name" in df2.columns:
        cols["src_ads_name"]   = df2.loc[mismatch_idx, "ads_name"].astype(str).values

    pd.DataFrame(cols).to_csv(out_csv, index=False, encoding="utf-8-sig")
    print(f"\n[저장] 불일치 리포트: {out_csv}")

else:
    print("✅ 결합된 domain_label이 라벨링 파일의 domain_label과 (순서/값) 완전히 동일합니다.")
print("=================================================")